In [2]:
import pandas as pd

In [3]:
from pathlib import Path

# Resolve o caminho dos dados a partir do root do workspace ou da pasta notebooks/
DATA_DIR = Path('data/raw')
if not DATA_DIR.exists():
    DATA_DIR = Path('../data/raw')

csv_files = sorted(DATA_DIR.glob('*.csv'))
print(f'{len(csv_files)} arquivos CSV em {DATA_DIR.resolve()}')
for p in csv_files:
    print(f'  - {p.name} ({p.stat().st_size / 1e6:.1f} MB)')

9 arquivos CSV em D:\bre_p\bre_p\Documents\projeto-tcc\data\raw
  - olist_customers_dataset.csv (9.0 MB)
  - olist_geolocation_dataset.csv (61.3 MB)
  - olist_order_items_dataset.csv (15.4 MB)
  - olist_order_payments_dataset.csv (5.8 MB)
  - olist_order_reviews_dataset.csv (14.5 MB)
  - olist_orders_dataset.csv (17.7 MB)
  - olist_products_dataset.csv (2.4 MB)
  - olist_sellers_dataset.csv (0.2 MB)
  - product_category_name_translation.csv (0.0 MB)


In [ ]:
def count_lines(path: Path, chunk_size: int = 1 << 20) -> int:
    """Conta linhas fisicas do arquivo em blocos, sem carrega-lo inteiro na memoria."""
    with path.open('rb') as f:
        total = 0
        last_byte = b''
        while chunk := f.read(chunk_size):
            total += chunk.count(b'\n')
            last_byte = chunk[-1:]
    return total + (1 if last_byte not in (b'\n', b'') else 0)


resumo = pd.DataFrame(
    {
        'arquivo': [p.name for p in csv_files],
        'linhas_total': [count_lines(p) for p in csv_files],
    }
)
# linhas de dados = linhas fisicas - header
resumo['linhas_dados'] = resumo['linhas_total'] - 1
resumo['% do total'] = (resumo['linhas_dados'] / resumo['linhas_dados'].sum() * 100).round(2)
resumo = resumo.sort_values('linhas_dados', ascending=False).reset_index(drop=True)

total = int(resumo['linhas_total'].sum())
print(f'Total de linhas (com headers): {total:,}')
print(f'Total de linhas de dados (sem headers): {total - len(resumo):,}')
resumo

Total de linhas (com headers): 1,556,426
Total de linhas de dados (sem headers): 1,556,417


,arquivo,linhas_total,linhas_dados,% do total
0,olist_geolocation_dataset.csv,1000164,1000163,64.26
1,olist_order_items_dataset.csv,112651,112650,7.24
2,olist_order_reviews_dataset.csv,104720,104719,6.73
3,olist_order_payments_dataset.csv,103887,103886,6.67
4,olist_customers_dataset.csv,99442,99441,6.39
5,olist_orders_dataset.csv,99442,99441,6.39
6,olist_products_dataset.csv,32952,32951,2.12
7,olist_sellers_dataset.csv,3096,3095,0.20
8,product_category_name_translation.csv,72,71,0.00


: 